# Load rekordbox XML Tracks Into Neo4j

## Load useful libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import xmltodict
import re

In [ ]:
from music_production_and_DJ_tools.theory.western.scales.pitch_class_scales.chromatic import pitch_class_chromatic_scale_camelot_system
from music_production_and_DJ_tools.theory.western.scales.pitch_class_scales.chromatic import pitch_class_chromatic_scale_opposite_of_camelot_system
from music_production_and_DJ_tools.dj.harmonic_mixing.camelot_wheel import CamelotWheel

In [ ]:
from python_tools_and_shortcuts.databases.neo4j.Neo4jInterface import Neo4jInterface

## Connect to the database

In [ ]:
nj = Neo4jInterface(os.environ['NEO4J_PASSWORD'])

## Map keys expressed as letters and accidentals to Camelot keys

In [ ]:
list_chromatic_scales = [
    pitch_class_chromatic_scale_camelot_system,
    pitch_class_chromatic_scale_opposite_of_camelot_system,
]

In [ ]:
c = CamelotWheel(list_chromatic_scales)

In [ ]:
c.df_initial_wheel_stacked.head(3)

In [ ]:
dict_key_to_camelot_key_map = {}
for i, row in c.df_initial_wheel_stacked.iterrows():
    dict_key_to_camelot_key_map[row['key'].strip()] = row['camelot_key'].strip()
for i, row in c.df_initial_wheel_stacked.iterrows():
    dict_key_to_camelot_key_map[row['camelot_key'].strip()] = row['camelot_key'].strip()    

## Load and parse the XML file

In [ ]:
filename_xml_export = os.environ['HOME'] + '/Desktop/export.xml'

In [ ]:
with open(filename_xml_export) as f:
    xml = f.read()

In [ ]:
dict_export = xmltodict.parse(xml)

## Construct DataFrame

In [ ]:
df = pd.DataFrame(dict_export['DJ_PLAYLISTS']['COLLECTION']['TRACK'])
df = df[(~df['@Artist'].isna()) & (df['@Artist'] != '')]
df = df[df['@Artist'] != 'Loopmasters']

columns_to_keep = [
    '@TrackID', '@Name', '@Artist', '@Album',
    '@Genre', '@TotalTime', 
    '@Year', '@AverageBpm', '@BitRate', '@SampleRate',
    '@Comments', '@Tonality',
]

df = df[columns_to_keep]

df.rename(
    columns = {
        '@TrackID' : 'rekordbox_track_id',
        '@Name' : 'track_name',
        '@Artist' : 'artist',
        '@Album' : 'album',
        '@Genre' : 'genre',
        '@TotalTime' : 'duration_in_seconds',
        '@Year' : 'year',
        '@AverageBpm' : 'bpm',
        '@BitRate' : 'bit_rate',
        '@SampleRate' : 'sample_rate',
        '@Tonality' : 'key',
    },
    inplace = True,
)

df['energy_str'] = [re.sub(' +', ' ', x.strip().split('Energy ')[-1].split(' ')[0]) for x in df['@Comments']]
df['energy'] = [np.int8(x) if x.isdigit() else np.nan for x in df['energy_str']]
df['energy'] = df['energy'].astype('Int64')

df['rekordbox_track_id'] = df['rekordbox_track_id'].astype('Int64')
df['duration_in_seconds'] = df['duration_in_seconds'].astype('Int64')
df['year'] = df['year'].astype('Int64')
df['bit_rate'] = df['bit_rate'].astype('Int64')
df['sample_rate'] = df['sample_rate'].astype('Int64')

df['bpm'] = df['bpm'].astype('Float64')

df['year'].replace(0, np.nan, inplace=True)

# temp
df = df[~df['genre'].isin(['FemDom', 'Sex - F', 'Sex - M', 'Fae', 'Kink', 'New', 'Cool', 'Test', "Children's Music"])]

df['genre'] = [x.lower() for x in df['genre']]



df = df[df['key'] != '']
df['key'] = [x.replace('♯', '#') for x in df['key']]
df['key'] = [dict_key_to_camelot_key_map[x] for x in df['key']]




df.drop(columns = ['energy_str', '@Comments'], inplace = True)

df = df.copy()

## QA #1

In [ ]:
df.head(3)

## Remove preceding and trailing whitespace from the strings

In [ ]:
for column in ['track_name', 'artist', 'album', 'genre', 'key']:
    df[column] = [x.strip() for x in df[column]]

## QA #2

In [ ]:
df.dtypes

## Define the node lists

In [ ]:
list_artists = [
    {'artist' : x.strip()} for x in sorted(
        list(
            df[df['artist'] != '']['artist'].unique()
        )
    ) if x.strip() != ''
]

In [ ]:
list_artists[0:3]

In [ ]:
list_albums = [
    {'album' : x.strip()} for x in sorted(
        list(
            df[df['album'] != '']['album'].unique()
        )
    ) if x.strip() != ''
]

In [ ]:
list_albums[0:3]

In [ ]:
list_genres = [
    {'genre' : x.strip()} for x in sorted(
        list(
            df[df['genre'] != '']['genre'].unique()
        )
    ) if x.strip() != ''
]

In [ ]:
list_genres[0:3]

In [ ]:
list_energy = [{'energy' : int(x)} for x in sorted(list([x for x in df['energy'].unique() if not pd.isna(x)]))]

In [ ]:
list_energy

In [ ]:
list_tracks = (
    df[(df['artist'] != '') & (df['track_name'] != '') & (~df['rekordbox_track_id'].isna())]
    [['rekordbox_track_id', 'track_name', 'duration_in_seconds', 'year', 'bpm']]
    .dropna()
    .to_dict(orient = 'records')
)

In [ ]:
list_tracks[0:3]

## Clear the nodes from the database

In [ ]:
nj.drop_everything_by_node_type('MUSIC_ARTIST')
nj.drop_everything_by_node_type('MUSIC_ALBUM')
nj.drop_everything_by_node_type('MUSIC_GENRE')
nj.drop_everything_by_node_type('MUSIC_ENERGY')
nj.drop_everything_by_node_type('MUSIC_TRACK')

## Insert nodes

In [ ]:
nj.drop_everything_by_node_type('MUSIC_ARTIST')

def batch_artist_merge(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_ARTIST {name: row.artist})
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_artist_merge, list_artists)

In [ ]:
nj.drop_everything_by_node_type('MUSIC_ALBUM')

def batch_album_merge(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_ALBUM {name: row.album})
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_album_merge, list_albums)

In [ ]:
nj.drop_everything_by_node_type('MUSIC_GENRE')

def batch_genre_merge(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_GENRE {name: row.genre})
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_genre_merge, list_genres)

In [ ]:
nj.drop_everything_by_node_type('MUSIC_ENERGY')

def batch_energy_merge(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_ENERGY {level: row.energy})
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_energy_merge, list_energy)

In [ ]:
nj.drop_everything_by_node_type('MUSIC_TRACK')

def batch_track_merge(tx, batch):
    query = """
    UNWIND $batch AS row
    MERGE (n1:MUSIC_TRACK {name: row.track_name, rekordbox_track_id: row.rekordbox_track_id, duration_in_seconds: row.duration_in_seconds, year: row.year, tempo: row.bpm})
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_track_merge, list_tracks)

## Insert relationships

In [ ]:
list_relationships_album_has_artist = (
    df[(df['artist'] != '') & (df['album'] != '')]
    [['artist', 'album']]
    .dropna()
    .to_dict(orient = 'records')
)

def batch_merge_album_has_artist(tx, batch):
    query = """
    UNWIND $batch AS row
    MATCH (ar:MUSIC_ARTIST {name: row.artist})
    MATCH (al:MUSIC_ALBUM {name: row.album})
    MERGE (al)-[r:ALBUM_HAS_ARTIST]->(ar)
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_merge_album_has_artist, list_relationships_album_has_artist)

In [ ]:
list_relationships_artist_has_track = (
    df[(df['artist'] != '') & (df['track_name'] != '') & (~df['rekordbox_track_id'].isna())]
    [['artist', 'rekordbox_track_id']]
    .dropna()
    .to_dict(orient = 'records')
)

def batch_merge_artist_has_track(tx, batch):
    query = """
    UNWIND $batch AS row
    MATCH (ar:MUSIC_ARTIST {name: row.artist})
    MATCH (t:MUSIC_TRACK {rekordbox_track_id: row.rekordbox_track_id})
    MERGE (ar)-[r:ARTIST_HAS_TRACK]->(t)
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_merge_artist_has_track, list_relationships_artist_has_track)

In [ ]:
list_relationships_track_has_album = (
    df[(df['album'] != '') & (df['track_name'] != '') & (~df['rekordbox_track_id'].isna())]
    [['album', 'rekordbox_track_id']]
    .dropna()
    .to_dict(orient = 'records')
)

def batch_merge_track_has_album(tx, batch):
    query = """
    UNWIND $batch AS row
    MATCH (al:MUSIC_ALBUM {name: row.album})
    MATCH (t:MUSIC_TRACK {rekordbox_track_id: row.rekordbox_track_id})
    MERGE (t)-[r:TRACK_HAS_ALBUM]->(al)
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_merge_track_has_album, list_relationships_track_has_album)

In [ ]:
list_relationships_track_has_genre = (
    df[(df['genre'] != '') & (df['track_name'] != '') & (~df['rekordbox_track_id'].isna())]
    [['genre', 'rekordbox_track_id']]
    .dropna()
    .to_dict(orient = 'records')
)

def batch_merge_track_has_genre(tx, batch):
    query = """
    UNWIND $batch AS row
    MATCH (g:MUSIC_GENRE {name: row.genre})
    MATCH (t:MUSIC_TRACK {rekordbox_track_id: row.rekordbox_track_id})
    MERGE (t)-[r:TRACK_HAS_GENRE]->(g)
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_merge_track_has_genre, list_relationships_track_has_genre)

In [ ]:
list_relationships_track_has_energy = (
    df[(df['track_name'] != '') & (~df['rekordbox_track_id'].isna())]
    [['energy', 'rekordbox_track_id']]
    .dropna()
    .to_dict(orient = 'records')
)

def batch_merge_track_has_energy(tx, batch):
    query = """
    UNWIND $batch AS row
    MATCH (e:MUSIC_ENERGY {level: row.energy})
    MATCH (t:MUSIC_TRACK {rekordbox_track_id: row.rekordbox_track_id})
    MERGE (t)-[r:TRACK_HAS_ENERGY]->(e)
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_merge_track_has_energy, list_relationships_track_has_energy)

In [ ]:
df_relationships_track_has_key = (
    pd.merge(
        (
            df[(df['key'] != '') & (df['track_name'] != '') & (~df['rekordbox_track_id'].isna())]
            [['rekordbox_track_id', 'key']]
            .dropna()
            .rename(columns = {'key' : 'camelot_key'})
        ),
        c.df_initial_wheel_stacked[['camelot_key', 'key']].sort_values(by = 'camelot_key'),
        on = 'camelot_key',
        how = 'left',
    )
    .drop(columns = ['camelot_key'])
)

list_relationships_track_has_key = df_relationships_track_has_key.dropna().to_dict(orient = 'records')

def batch_merge_track_has_key(tx, batch):
    query = """
    UNWIND $batch AS row
    MATCH (k:MUSIC_KEY {key: row.key})
    MATCH (t:MUSIC_TRACK {rekordbox_track_id: row.rekordbox_track_id})
    MERGE (t)-[r:TRACK_HAS_KEY]->(k)
    """
    tx.run(query, batch=batch)

nj.batch_it(batch_merge_track_has_key, list_relationships_track_has_key)